# Сложные операции над объектами Pandas

На прошлом занятии мы изучили простые операции над объектами Pandas: векторизованные операции, операции для очистки данных, а также операции, которые могут быть полезны для преобразования данных в отдельных столбцах. В этом занятии мы рассмотрим более комплексные операции над объектами Pandas. Эти операции включают в себя объединение датафреймов по индексам и колонкам, а также агрегирование данных.

**Необходимые импорты**:

In [ ]:
from collections.abc import Sequence
from typing import Any
from uuid import uuid4

import numpy as np
import pandas as pd

**Вспомогательные объекты**:

In [ ]:
def create_df_from_index_and_columns(
    index: Sequence[Any],
    columns: Sequence[Any],
) -> pd.DataFrame:
    data = {
        column_name: [
            f"{index_name}|{column_name}" for index_name in index
        ]
        for column_name in columns
    }

    return pd.DataFrame(data, index=index)

## Объединение данных

Очень часто при решении практических задач вам придется иметь дело не с одним единственным источником данных, а с несколькими. Данные могут храниться в разных csv-файлах, таблицах в БД и даже на разных серверах. Поэтому важно уметь правильно объединять данные из разных источников в единый датафрейм для удобства дальнейшей работы (конечно, если это целесообразно в рамках решаемой задачи). Итак, в этой секции рассмотрим наиболее распространенные способы объединения данных в Pandas.

### Конкатенация

Конкатенация - это простейшая форма объединения двух и более наборов данных в единый набор. Несмотря на свою простоту, конкатенация бывает очень полезна в тех случаях, когда вам нужно объединить данные с одинаковой структурой, полученные из разных источников. Примером таких данных может быть некоторая таблица, которая хранится в виде набора csv-файлов с одинаковой структурой. Чтобы восстановить исходную таблицу из данного набора csv-файлов, нам достаточно просто прочитать содержимое этих файлов и выполнить конкатенацию полученных датафреймов.

Конкатенация в Pandas напоминает конкатенацию в NumPy и выполняется с помощью функции `pd.concat`. Обязательный аргумент функции `pd.concat` - итерируемый объект `objs` с объектами, которые требуется объединить. С помощью этой функции вы можете объединить два и более набора данных (`pd.Series` или `pd.DataFrame`), записав данные из них последовательно, друг за другом.

Простейший вариант использования функции `pd.concat` - конкатенация двух объектов `pd.Series`:

In [ ]:
series_size = 3

series1 = pd.Series(
    np.random.randint(170, 190, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series2 = pd.Series(
    np.random.randint(150, 180, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series_concatenated = pd.concat((series1, series2))

print(
    f"series1:\n{series1}",
    f"series2:\n{series2}",
    f"concatenation result:\n{series_concatenated}",
    sep="\n\n",
)

Функция `pd.concat` также может быть использована для объединения более сложных объектов, типа `pd.DataFrame`:

In [ ]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=[1, 2],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=[3, 4],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

По умолчанию объединение объектов `pd.Dataframe` с помощью `pd.concat` выполняется построчно. Однако, при объединении данных высокой размерности `pd.concat` предоставляет возможность настройки, как именно будут объединены данные. За эту настройку отвечает знакомый нам аргумент `axis`. Этот аргумент может быть связан как с целочисленными значениями, как мы уже видели при изучении NumPy, так и со строковыми литералами, типа `"columns"`. Использование строковых литералов предпочтительнее из-за увеличения понятности кода.

In [ ]:
index = ["row1", "row2"]

dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=index,
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=index,
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    axis="columns",
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

Во всех предыдущих примерах мы предполагали, что объединяемые объекты имеют или одни и те же индексы, или одни и те же имена столбцов. Однако `pd.concat` будет корректно работать и в тех случаях, когда это не так. Если объединяемые данные имеют разный набор индексов и столбцов, место недостающих данных будет заполнено `NaN` значениями.

In [ ]:
dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=["row3", "row4"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

#### Борьба с дублированием индексов

После объединения набора данных функция `pd.concat` сохраняет исходные индексы. Данное поведение может приводить к появлению дублирующихся индексов.

В приведенном примере результирующий `pd.DataFrame` будет содержать два значения `"row2"` в своем индексе.

In [ ]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

В некоторых ситуациях такое дублирование нежелательно. `pd.concat` предоставляет два способа по борьбе с дублирующимися индексами: игнорирование индексов исходных наборов данных и проверка уникальности.

При игнорировании индексов исходных наборов данных результат конкатенации будет содержать новый индекс, сгенерированный автоматически. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `ignore_index=True`.

In [ ]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    ignore_index=True,
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

В этом случае индекс не будет содержать никаких дубликатов, однако при этом вы теряете значения исходных индексов. Поэтому данный способ борьбы с дубликатами подходит в том случае, если значения индекса неважны для решения задачи.

Второй способ борьбы с дубликатами - проверка уникальности. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `verify_integrity=True`.

In [ ]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
# этот вызов должен привести к ошибке ValueError
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    verify_integrity=True,
)

При использовании флага `verify_integrity=True` вызов функции `pd.concat` завершится успехом в том и только в том случае, если индексы исходных наборов данных не имели пересечения. В противном случае вы получите ошибку `ValueError`.

При использовании данного подхода в результате конкатенации сохраняются значения индексов исходных наборов данных.

Существует еще один способ борьбы с дубликатами - создание мульти-индекса. Однако этот способ в нашем курсе рассматриваться не будет (как и мульти-индексы). Если вы заинтересовались темой мульти-индексов, вы можете ознакомиться с ней самостоятельно в [официальной документации](https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.MultiIndex.html).

### Сложные объединения данных

Сложные объединения данных в Pandas являются реализациями правил реляционной алгебры по объединению данных. Звучит страшно и сложно, но на практике все не так пугающе, как кажется на первый взгляд. Те из вас, кто знаком с SQL, увидят в описанных функция операцию `join`.

Сложные объединения данных незаменимы в тех случаях, когда данные, полученные из разных источников (например, из разных файлов), имеют сложные системы связей. Например, у нас может быть датасет с данными о пользователях и отдельный датасет с данными об активности пользователей на нашем сайте. Сами по себе датасеты имеют разную структуру и связаны между собой через значения колонки `user_id`. В этом случае для комплексного анализа данных об активности пользователей нашего сайта нам потребуется сложное объединение этих данных, а не простая конкатенация.

В Pandas сложное объединение наборов данных может быть выполнено с помощью функции `pd.merge`.

Чтобы продемонстрировать возможности функции `pd.merge`, рассмотрим пример. Предположим мы занимаемся анализом данных о покупках пользователей в нашем онлайн магазине. Данные хранятся в отдельных csv-файлах:
- [user](./data/user.csv) - данные о пользователях. В этом файле хранятся уникальные ID пользователей, а также персональные данные: username и email.
- [item](./data/item.csv) - данные о товарах. В этом файле хранятся ID товаров, их наименования и стоимости.
- [order_to_user](./data/order_to_user.csv) - связи между пользователями и ID заказов, которые эти пользователи совершали.
- [order_to_item](./data/order_to_item.csv) - связи между заказами и ID позиций, которые входят в заказ.

Загрузим эти данные и посмотрим на них:

In [ ]:
user_data = pd.read_csv("./data/user.csv")
item_data = pd.read_csv("./data/item.csv")
order_to_user = pd.read_csv("./data/order_to_user.csv")
order_to_item = pd.read_csv("./data/order_to_item.csv")

print("user data:")
display(user_data)

print("item data:")
display(item_data)

Итак, чтобы получить полные данные, которые бы включали в себя информацию о том, какой покупатель приобрел какую позицию в нашем магазине, нам необходимо объединить разрозненные данные. Очевидно, что все датафреймы имеют разную структуру, поэтому простая конкатенация нам не поможет. Однако датафреймы связаны между собой следующим образом:
- Датафрейм `user_data` связана с датафреймом `order_to_user` через столбец `user_id`.
- Датафрейм `item_data` связана с датафреймом `order_to_item` через столбец `item_id`.
- Датафрейм `order_to_user` связана с датафреймом `order_to_item` через столбец `order_id`.

Используя эти связи, мы можем объединить данные в один датафрейм с помощью функции `pd.merge`:

In [ ]:
order_to_item_expended = pd.merge(order_to_item, item_data)
order_to_user_extended = pd.merge(order_to_user, user_data)

pd.merge(order_to_user_extended, order_to_item_expended)

За один раз функция `pd.merge` может объединить только два `pd.DataFrame`, поэтому в примере выше нам пришлось вызвать ее три раза. По умолчанию эта функция выполняет поиск столбцов с одинаковыми именами и использует значения этих столбцов в качестве ключей слияния. Проще говоря по умолчанию `pd.merge` делает следующее:
- Находит столбцы с одинаковыми именами в правом и в левом `pd.DataFrame`.
- Для каждой строки левого `pd.DataFrame` ищет строку в правом `pd.DataFrame` такую, что в столбцах, найденных на предыдущем шаге, находятся одинаковые значения.
- Если такая строка найдена, то строка из левого `pd.DataFrame` объединяется со строкой из правого `pd.DataFrame`. Результирующая строка сохраняется в результирующий `pd.DataFrame`.

Однако, к сожалению, на практике далеко не всегда удается использовать дефолтное поведение `pd.merge`. В первую очередь это связано с неоднородностью данных: столбцы с одинаковыми данными в разных `pd.DataFrame` могут иметь разные названия.

In [ ]:
order_to_user = order_to_user.rename(columns={"user_id": "UserID"})
# этот вызов приведет к ошибке MergeError,
# т.к. датафреймы не содержат общих колонок
pd.merge(user_data, order_to_user)

В данном примере датафрейм `user_data` содержит столбец `user_id`, а датафрейм `order_to_user` - `UserID`. Поскольку датафреймы не имеют общих столбцов, использовать поведение `pd.merge` по умолчанию невозможно.

На этот случай `pd.merge` позволяет явно указать, какой столбец левого датафрейма и какой столбец правого датафрейма стоит использовать в качестве ключей слияния. Чтобы это сделать, необходимо связать имена столбцов слияния левого и правого датафреймов с аргументами `left_on` и `right_on`, соответственно:

In [ ]:
pd.merge(
    user_data,
    order_to_user,
    left_on="user_id",
    right_on="UserID",
)

В результирующий датафрейм будут включены оба столбца. Если вас не устраивает подобное поведение, вы можете явно избавиться от нежелательного столбца с помощью метода `pd.DataFrame.drop`.

В тех случаях, когда датафреймы содержат информативный индекс, `pd.merge` позволяет выполнить слияние по индексам. Для этого достаточно передать флаг `left_index=True`, если в качестве ключа слияния для левого датафрейма следует использовать его индекс, и `right_index=True`, если в качестве ключа слияния для правого датафрейма нужно использовать его индекс.

In [ ]:
pd.merge(
    item_data,
    order_to_item,
    left_index=True,
    right_index=True,
)

Для упрощения слияния по индексам в объектах `pd.DataFrame` и `pd.Series` реализован метод `join`.

При необходимости вы можете комбинировать ключи слияния: использовать в качестве ключа слияния для одного датафрейма индекс, а для второго - столбец:

In [ ]:
user_data.set_index("user_id", inplace=True)
pd.merge(
    user_data,
    order_to_user,
    left_index=True,
    right_on="UserID",
)

По умолчанию при объединении данных с помощью `pd.merge` результирующий датафрейм будет содержать только те строки, для которых совпали значения ключей слияния. Если для какой-то строки одного из операндов не было найдено строки с соответствующим значением ключа слияния в другом операнде, такая строка включена в результат не будет.

In [ ]:
dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col2", "col3"],
    index=["row2", "row3"]
)
dataframe_merged = pd.merge(dataframe1, dataframe2)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"dataframe merged:\n{dataframe_merged}",
    sep="\n\n",
)

С одной стороны, это логичное поведение. С другой же стороны, на практике это поведение не всегда может быть удобным. Для выбора стратегии объединения данных необходимо использовать параметр `how` функции `pd.merge`. По умолчанию значение этого параметра `how="inner"`.

Помимо уже рассмотренного внутреннего объединения, существую следующие стратегии объединения:
- `outer` - в результат включаются все строки из обоих операндов. Отсутствующие данные заполняются значениями `NaN`:

In [ ]:
pd.merge(dataframe1, dataframe2, how="outer")

- `left` - в результат включаются все строки из левого операнда. Отсутствующие данные заполняются значениями `NaN`:

In [ ]:
pd.merge(dataframe1, dataframe2, how="left")

- `right` - в результат включаются все строки из правого операнда. Отсутствующие данные заполняются значениями `NaN`:

In [ ]:
pd.merge(dataframe1, dataframe2, how="right")

## Группировка и агрегирование

В простом объединении данных из разных источников нет особого смысла. Обычно данные из разных источников объединяют, чтобы каким-то образом их сгруппировать, а затем вычислить некоторые сводные показатели. Для этих целей к наборам данных применяют следующие операции: объединение, группировка, агрегирование. В данный момент мы рассмотрели, как в Pandas выполняется объединение данных. Теперь давайте рассмотрим на оставшиеся две операции.

In [ ]:
user_data = pd.read_csv("./data/user.csv")
item_data = pd.read_csv("./data/item.csv")
order_to_user = pd.read_csv("./data/order_to_user.csv")
order_to_item = pd.read_csv("./data/order_to_item.csv")

order_to_item_expended = pd.merge(order_to_item, item_data)
order_to_user_extended = pd.merge(order_to_user, user_data)
dataframe = pd.merge(order_to_user_extended, order_to_item_expended)
dataframe.loc[:, "price"] = (
    dataframe.loc[:, "price"].str.lstrip("$")
    .astype(float)
)
dataframe.sample(3)

### Группировка данных

Итак, вернемся к нашему примеру с онлайн магазином. Мы объединили все разрозненные данные и теперь хотим их проанализировать: вичислить некоторые сводные показатели:
- Сколько покупок совершил каждый пользователь.
- Какой доход каждый пользователь принес нашему магазину.
- Средняя стоимость заказа.

Чтобы подсчитать все эти характеристики, нам потребуется научиться группировать исходные данные по пользователям и по заказам. Для этого нам подойдет метод `groupby` объектов `pd.DataFrame`:

In [ ]:
dataframe_by_user_id = dataframe.groupby(by="user_id")
print(dataframe_by_user_id)

Результат выполнения метода `groupby` - объект типа `DataFrameGroupBy`. Это некоторый полуфабрикат, промежуточный объект, который содержит сгруппированные данные и позволяет эффективно агрегировать данные в рамках отдельных групп.

При необходимости вы можете посмотреть, какие группы содержит полученный объект, и какие строки включены в эти группы:

In [ ]:
for group_key, row_ids in dataframe_by_user_id.groups.items():
    print(f"{group_key}: {row_ids}")

Также вы можете получить датафрейм, соответствующий конкретной группе. Для этого достаточно воспользоваться методов `get_group`:

In [ ]:
dataframe_by_user_id.get_group("30f0dff3-8ba3-40c4-a62f-821c0ed09784")

### Агрегирование данных

Однако сам по себе объект `DataFrameGroupBy` не так интересен. Гораздо интереснее возможность быстрого подсчета некоторых показателей для каждой каждой группы. Для этого достаточно использовать знакомые нам со времен NumPy методы агрегации.

Давайте подсчитаем доход, полученный от каждого пользователя, используя объект `dataframe_by_user_id`:

In [ ]:
dataframe_by_user_id["price"].sum().sort_values(ascending=False)

Давайте также подсчитаем, сколько товаров приобрел каждый пользователь:

In [ ]:
dataframe_by_user_id["item_id"].count()

Сколько заказов сделал каждый пользователь:

In [ ]:
dataframe_by_user_id["order_id"].nunique()

Средняя стоимость заказа:

In [ ]:
dataframe_by_order_id = dataframe.groupby("order_id")
dataframe_by_order_id["price"].sum().mean()